In [2]:
# importing the packages
import numpy as np
import pandas as pd
from itertools import chain
from ortools.linear_solver import pywraplp
from warnings import filterwarnings

filterwarnings("ignore")

# # create a zero np_array
# pair_matrix = np.zeros((len(pairings), len(np_arr)))

# # fill the values of the flight legs in each pairing with 1
# for i, pair in enumerate(pairings):
#     pair_matrix[i, list(chain.from_iterable(pair))] = 1

In [3]:
# Reading the dataframe and converting it to a numpy array
np_arr = pd.read_csv(
    "../data/flight_legs/data.csv",
    parse_dates=["start_time", "end_time"],
).to_numpy()


# read the duties from the txt file
with open("../data/pairings/pairings.txt") as file:
    pairings = [list(chain.from_iterable(eval(line))) for line in file]


# calculate the cost matrix from pair list
cost_matrix = np.array(
    [
        (np_arr[pair[-1]][4] - np_arr[pair[0]][3]).total_seconds() / 3600
        for pair in pairings
    ]
).reshape(-1, 1)

# determining the number of flights and tasks
num_pairs = len(pairings)
num_flights = len(np_arr)
print(num_pairs, num_flights)

82092 108


In [4]:
# Initializing the MIP Solver
solver = pywraplp.Solver.CreateSolver("GLOP")

# creating the binary allocation variable
x = np.array([solver.BoolVar("") for i in range(num_pairs)]).reshape(-1, 1)
# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
for i in range(num_flights):
    solver.Add(
        solver.Sum([x[j][0] * 1.0 for j in range(num_pairs) if i in pairings[j]]) == 1.0
    )
# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i][0] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i][0].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

The Solution is OPTIMAL
82092
[1262, 1266, 1287, 2573, 3515, 3538, 3593, 4087, 4748, 5042, 5964, 6696, 7022, 7071, 7091, 15660, 15835, 15876, 16004, 16020, 18003, 23007, 23695, 23953, 29680, 29988, 30316, 30415, 30475, 30647, 34473, 34671, 36252, 38748, 42224, 42294, 42500, 42501, 43882, 48422, 48429, 49545, 50005, 61004, 61081, 61155, 61159, 61197, 61200, 61515, 64082, 69245, 70389, 70393, 78674, 78950, 78964]
643.0000000000002
[21.999999999999886, -1.0000000000000764, 3.0000000000000937, 19.999999999999762, 6.000000000000092, 0.9999999999999876, 0.0, 13.000000000000023, 9.000000000000023, -2.7755575615628914e-14, 3.000000000000098, 21.99999999999996, 3.000000000000038, 21.999999999999957, 0.0, 20.00000000000003, -16.999999999999936, 18.9999999999999, 21.999999999999726, 1.0000000000000262, 3.0000000000000693, 7.99999999999984, 18.000000000000025, 2.999999999999996, 1.9999999999999751, 3.9968028886505635e-14, 21.999999999999993, 1.999999999999973, 3.0000000000000924, 21.99999999999997

In [5]:
# Initializing the MIP Solver
solver = pywraplp.Solver.CreateSolver("SAT")

# creating the binary allocation variable
x = np.array([solver.BoolVar("") for i in range(num_pairs)]).reshape(-1, 1)
# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
for i in range(num_flights):
    solver.Add(
        solver.Sum([x[j][0] * 1.0 for j in range(num_pairs) if i in pairings[j]]) == 1.0
    )
# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i][0] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i][0].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

The Solution is OPTIMAL
82092
[2573, 3515, 5922, 16020, 23003, 34602, 42443, 42501, 48528, 61144, 70551, 78599, 78950]
643.0
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


: 

In [5]:
# importing the packages
import numpy as np
from ortools.linear_solver import pywraplp

# Initialize the solver
solver = pywraplp.Solver.CreateSolver("CBC")
use_dual_simplex: True

# Decision Variable
x = [solver.NumVar(0, 1, f"x_{i}") for i in range(num_pairs)]

# Constraints
for j in range(num_flights):
    solver.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) - 1 >= 0)

# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
x_values = [x[i].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

NameError: name 'pair_matrix' is not defined

In [6]:
# importing the packages
import numpy as np
from ortools.sat.python import cp_model

# Declare the CP_SAT model
model = cp_model.CpModel()

# create the decision variable
x = []
for i in range(num_pairs):
    x.append(model.NewBoolVar(f"x_{i}"))

# Constraints
constraints = []
for j in range(num_flights):
    constraints.append(
        model.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) >= 1)
    )

# Objective Function
model.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the Problem
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
    selected_pairs = [i for i in range(num_pairs) if solver.Value(x[i]) == 1]
    print(selected_pairs)
else:
    print("oops")

print(solver.ObjectiveValue(), len(selected_pairs))

NameError: name 'pair_matrix' is not defined